In [ ]:
# =============================================================================
# EXPLAINABLE FAKE REVIEW DETECTION USING ENSEMBLE MACHINE LEARNING AND SHAP
# Final merged upgrade — paste each "# %%" section as a separate notebook cell
# =============================================================================

# %% [1] INSTALL / IMPORTS
# -----------------------------------------------------------------------------
!pip install -q nltk shap gradio openpyxl

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import string
import pickle

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from scipy.sparse import hstack
from wordcloud import WordCloud

from google.colab import files

from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (RandomForestClassifier, AdaBoostClassifier,
                               VotingClassifier, StackingClassifier)
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier

from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, roc_curve,
                              confusion_matrix, classification_report)

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

import shap
import gradio as gr




In [ ]:
# %% [2] LOAD DATA
# -----------------------------------------------------------------------------
PATH_DATASET = "/content/fake_reviews_dataset.csv"

df = pd.read_csv(PATH_DATASET)
df.rename(columns={'text_': 'text'}, inplace=True)
df.head()

In [ ]:
def encode_label(df):
    labels = {'CG': 0, 'OR': 1}   # Create a mapping: CG = 0 (Fake), OR = 1 (Real)
    df['target'] = df['label'].map(labels)   # Convert text labels into numeric values
    return df   # Return the updated DataFrame

df = encode_label(df)   # Apply label encoding to the dataset

df['target'] = df['target'].fillna(0).replace([np.inf, -np.inf], 0).astype(int)   # Replace missing/infinite values and convert to integer

df = df.sample(frac=1).reset_index(drop=True)   # Randomly shuffle all rows and reset the index

df.drop_duplicates(inplace=True)   # Remove duplicate rows from the dataset

df.isnull().sum()   # Count the number of missing (null) values in each column

In [ ]:
# %% [4] EDA — CLASS BALANCE
# -----------------------------------------------------------------------------
plt.pie(
    df['target'].value_counts(),                  # Count the number of samples in each class
    labels=['Fake (CG)', 'Real (OR)'],            # Labels for the pie chart
    autopct="%0.2f"                               # Display percentages with 2 decimal places
)

plt.title("Class Distribution")                  # Add a title to the chart

plt.savefig('class_distribution.png',            # Save the chart as a PNG image
            bbox_inches='tight',                 # Remove extra whitespace around the figure
            dpi=300)                             # Save in high resolution (300 DPI)

plt.show()                                       # Display the pie chart

# files.download('class_distribution.png')       # Download the saved image (uncomment to use)

In [ ]:
# %% [5] FEATURE ENGINEERING — KEEP RATING + CATEGORY + METADATA
# -----------------------------------------------------------------------------

le_category = LabelEncoder()   # Create a LabelEncoder object for categorical data

df['category_encoded'] = le_category.fit_transform(df['category'].astype(str))   # Convert product categories into numeric values

df['rating'] = df['rating'].astype(float)   # Convert the rating column to float data type

df['num_characters'] = df['text'].apply(len)   # Count the number of characters in each review

df['num_words'] = df['text'].apply(lambda x: len(nltk.word_tokenize(x)))   # Count the number of words in each review

df['num_sentences'] = df['text'].apply(lambda x: len(nltk.sent_tokenize(x)))   # Count the number of sentences in each review

df['capital_ratio'] = df['text'].apply(lambda x: sum(1 for c in x if c.isupper()) / (len(x) + 1))   # Calculate the ratio of uppercase letters

df['exclamation_count'] = df['text'].apply(lambda x: x.count('!'))   # Count the number of exclamation marks in each review

stats_df = df.describe()   # Generate descriptive statistics for all numerical columns

display(stats_df)   # Display the statistics table in the notebook

stats_df.to_csv('dataset_statistics.csv')   # Save the statistics table as a CSV file

files.download('dataset_statistics.csv')   # Download the CSV file to your computer

In [ ]:
# %% [6] EDA — CORRELATION OF ENGINEERED FEATURES
# -----------------------------------------------------------------------------

df_numeric = df.select_dtypes(include='number')   # Select only numeric columns for correlation analysis

plt.figure(figsize=(8, 6))   # Create a figure with a size of 8 × 6 inches

sns.heatmap(df_numeric.corr(), annot=True, cmap='coolwarm')   # Compute and visualize the correlation matrix

plt.title("Feature Correlation Heatmap")   # Add a title to the heatmap

plt.savefig('correlation_heatmap.png', bbox_inches='tight', dpi=300)   # Save the heatmap as a high-resolution image

plt.show()   # Display the heatmap

# Interpretation:
# Correlation values range from -1 to +1.
# Values close to +1 indicate a strong positive relationship.
# Values close to -1 indicate a strong negative relationship.
# Values near 0 indicate little or no linear relationship between features.
# This heatmap helps identify highly correlated features, which may be redundant
# and useful for feature selection or understanding relationships in the dataset.

# files.download('correlation_heatmap.png')   # Download the heatmap image

In [ ]:
# %% [7] TEXT PREPROCESSING — LEMMATIZATION
# -----------------------------------------------------------------------------

lemmatizer = WordNetLemmatizer()   # Create a lemmatizer object to convert words to their base form

stop_words = set(stopwords.words('english'))   # Load the English stopword list

def transform_text(text):
    text = text.lower()   # Convert all text to lowercase

    tokens = nltk.word_tokenize(text)   # Split the text into individual words (tokens)

    tokens = [t for t in tokens if t.isalnum()]   # Keep only letters and numbers, remove special characters

    tokens = [t for t in tokens if t not in stop_words and t not in string.punctuation]   # Remove stopwords and punctuation

    tokens = [lemmatizer.lemmatize(t) for t in tokens]   # Convert each word to its base (dictionary) form

    return " ".join(tokens)   # Join the processed words back into a single string

df['transformed_text'] = df['text'].apply(transform_text)   # Apply text preprocessing to every review

df.head()   # Display the first 5 rows of the updated dataset

df.to_csv('preprocessed_reviews.csv', index=False)   # Save the preprocessed dataset as a CSV file

# files.download('preprocessed_reviews.csv')   # Download the preprocessed dataset to your computer

In [ ]:
# %% [8] WORDCLOUDS
# -----------------------------------------------------------------------------
# Create a WordCloud object with custom settings.
# width, height      -> Size of the generated image
# min_font_size      -> Smallest font size for displayed words
# background_color   -> Background color of the word cloud
wc = WordCloud(width=500, height=500, min_font_size=10, background_color='white')


# =========================
# Fake Reviews Word Cloud
# =========================

# Select all fake reviews (target = 0),
# combine them into one long string using str.cat(),
# then generate the word cloud.
fake_wc = wc.generate(
    df[df['target'] == 0]['transformed_text'].str.cat(sep=" ")
)

# Create a new figure with a size of 10x5 inches.
plt.figure(figsize=(10, 5))

# Display the generated word cloud image.
plt.imshow(fake_wc)

# Hide x-axis and y-axis.
plt.axis('off')

# Add a title to the plot.
plt.title("Fake (CG) Reviews - Word Cloud")

# Save the figure as a high-quality PNG image.
# bbox_inches='tight' removes extra white space.
# dpi=300 gives high resolution.
plt.savefig('fake_wordcloud.png', bbox_inches='tight', dpi=300)

# Show the plot.
plt.show()

# Download the saved image (Google Colab only).
files.download('fake_wordcloud.png')


# =========================
# Real Reviews Word Cloud
# =========================

# Select all real reviews (target = 1),
# combine them into one string,
# then generate the word cloud.
real_wc = wc.generate(
    df[df['target'] == 1]['transformed_text'].str.cat(sep=" ")
)

# Create a new figure.
plt.figure(figsize=(10, 5))

# Display the generated word cloud.
plt.imshow(real_wc)

# Hide axes.
plt.axis('off')

# Add title.
plt.title("Real (OR) Reviews - Word Cloud")

# Save the figure as a PNG image.
plt.savefig('real_wordcloud.png', bbox_inches='tight', dpi=300)

# Display the figure.
plt.show()

# Download the saved image (Google Colab only).
# files.download('real_wordcloud.png')

In [ ]:
# %% [9] TF-IDF WITH BIGRAMS + FEATURE FUSION
# -----------------------------------------------------------------------------

# Create a TF-IDF Vectorizer.
# max_features=5000      -> Keep the 5000 most important words/features.
# ngram_range=(1,2)      -> Use both unigrams (single words) and bigrams (two-word combinations).
# sublinear_tf=True      -> Apply logarithmic scaling to term frequency,
#                           reducing the influence of very frequent words.
tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    sublinear_tf=True
)

# Learn the vocabulary from the transformed review text
# and convert each review into a TF-IDF feature vector.
X_text = tfidf.fit_transform(df['transformed_text'])


# ----------------------------
# Additional Metadata Features
# ----------------------------

# List of handcrafted numerical features to combine with TF-IDF features.
meta_cols = [
    'rating',              # Review rating (e.g., 1–5 stars)
    'category_encoded',    # Encoded product category
    'num_characters',      # Total number of characters
    'num_words',           # Total number of words
    'num_sentences',       # Total number of sentences
    'capital_ratio',       # Ratio of uppercase letters
    'exclamation_count'    # Number of exclamation marks
]


# ----------------------------
# Feature Scaling
# ----------------------------

# MinMaxScaler scales all numerical features to the range [0,1].
# This prevents large-valued features from dominating smaller ones
# and ensures compatibility with Multinomial Naive Bayes
# (which cannot handle negative feature values).
scaler = MinMaxScaler()

# Learn the minimum and maximum values of each metadata feature
# and transform them into the range [0,1].
X_meta = scaler.fit_transform(df[meta_cols])


# ----------------------------
# Feature Fusion
# ----------------------------

# Combine sparse TF-IDF features with scaled metadata features
# into one final feature matrix.
# hstack() performs a horizontal concatenation.
# tocsr() converts the result into Compressed Sparse Row format
# for faster computation and lower memory usage.
X = hstack([X_text, X_meta]).tocsr()


# Extract the target labels.
# 0 = Fake Review
# 1 = Genuine Review
y = df['target'].values


# Display the dimensions of the final feature matrix.
# (rows = reviews, columns = total features)
print("Final feature matrix shape:", X.shape)


# ----------------------------
# Train-Test Split
# ----------------------------

# Split the dataset into training (80%) and testing (20%).
# random_state=32 ensures reproducible results.
# stratify=y preserves the class distribution in both sets.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=32,
    stratify=y
)

In [ ]:
# %% [10] MODELS
# -----------------------------------------------------------------------------
knc = KNeighborsClassifier()
mnb = MultinomialNB()
dtc = DecisionTreeClassifier(max_depth=5)
lrc = LogisticRegression(solver='liblinear', penalty='l1', max_iter=1000)
rfc = RandomForestClassifier(n_estimators=100, random_state=2)
abc = AdaBoostClassifier(n_estimators=50, random_state=2)
xgb = XGBClassifier(n_estimators=100,   eval_metric='logloss')

clfs = {'KN': knc, 'NB': mnb, 'DT': dtc, 'LR': lrc, 'RF': rfc, 'AdaBoost': abc, 'XGB': xgb}



In [ ]:
# %% [11] CROSS-VALIDATION
# -----------------------------------------------------------------------------

# Import Google Colab file handling library.
# This is used to download generated files from Colab to the local computer.
from google.colab import files


# Print section heading.
print("5-Fold Cross-Validation (Accuracy):\n")


# Create an empty list to store cross-validation results
# for each machine learning algorithm.
cv_results = []


# Loop through every classifier stored in the dictionary 'clfs'.
# name -> model name (e.g., Logistic Regression, SVM, Naive Bayes)
# clf  -> actual machine learning model object
for name, clf in clfs.items():

    try:

        # Perform 5-Fold Cross-Validation.
        #
        # The training data is divided into 5 equal parts (folds).
        # The model trains on 4 folds and validates on 1 fold.
        # This process repeats 5 times, with each fold used once as validation.
        #
        # scoring='accuracy' calculates the percentage of correct predictions.
        scores = cross_val_score(
            clf,
            X_train,
            y_train,
            cv=5,
            scoring='accuracy'
        )


        # Calculate the average accuracy across all 5 folds.
        mean_score = scores.mean()


        # Calculate how much the accuracy varies between folds.
        # Lower standard deviation means the model performance is more stable.
        std_score = scores.std()


        # Display model performance.
        # :.4f displays accuracy up to 4 decimal places.
        print(f"{name}: {mean_score:.4f} (+/- {std_score:.4f})")


        # Store model name, average accuracy, and variation
        # in dictionary format for creating a table later.
        cv_results.append({
            'Algorithm': name,
            'Mean Accuracy': round(mean_score, 4),
            'Std Deviation': round(std_score, 4)
        })


    # If any model fails during cross-validation,
    # catch the error and continue testing other models.
    except Exception as e:
        print(f"{name}: skipped ({e})")


# Convert the list of dictionaries into a Pandas DataFrame.
# This creates a structured table containing all model results.
cv_df = pd.DataFrame(cv_results)


# Define the output CSV filename.
cv_filename = 'cross_validation_accuracy.csv'


# Save the cross-validation results as a CSV file.
# index=False prevents saving the DataFrame row numbers.
cv_df.to_csv(
    cv_filename,
    index=False
)


# Download the generated CSV file to the user's computer.
# Works in Google Colab environment.
# files.download(cv_filename)

In [ ]:
# %% [12] HYPERPARAMETER TUNING — LR AND RF ONLY
# -----------------------------------------------------------------------------

# ============================
# Logistic Regression Tuning
# ============================

# Define the hyperparameter search space for Logistic Regression.
#
# C:
#   Controls regularization strength.
#   Smaller C -> stronger regularization (simpler model).
#   Larger C -> weaker regularization (more complex model).
#
# penalty:
#   Type of regularization:
#   l1 -> Lasso regularization (can remove less important features)
#   l2 -> Ridge regularization (reduces feature weights)
lr_grid = {
    'C': [0.1, 1, 10],
    'penalty': ['l1', 'l2']
}


# Create GridSearchCV object.
#
# GridSearchCV automatically trains multiple models
# using every possible combination of parameters
# and selects the best-performing combination.
#
# solver='liblinear':
#   Supports both L1 and L2 regularization.
#
# max_iter=1000:
#   Maximum number of optimization iterations.
#
# cv=3:
#   Uses 3-fold cross-validation during parameter search.
#
# scoring='f1_weighted':
#   Selects the model with the highest weighted F1-score.
#   Useful when classes are not perfectly balanced.
#
# n_jobs=-1:
#   Uses all available CPU cores to speed up training.
lr_search = GridSearchCV(
    LogisticRegression(
        solver='liblinear',
        max_iter=1000
    ),
    lr_grid,
    cv=3,
    scoring='f1_weighted',
    n_jobs=-1
)


# Train GridSearchCV on training data.
# It tests all parameter combinations and finds the best one.
lr_search.fit(X_train, y_train)


# Extract the best Logistic Regression model
# with optimized hyperparameters.
lrc = lr_search.best_estimator_


# Display the best parameter combination found.
print("Best LR params:", lr_search.best_params_)



# ============================
# Random Forest Tuning
# ============================

# Define hyperparameter search space for Random Forest.
#
# n_estimators:
#   Number of decision trees in the forest.
#   More trees usually improve performance but increase computation.
#
# max_depth:
#   Maximum depth of each decision tree.
#   None means trees can grow until all leaves are pure.
rf_grid = {
    'n_estimators': [100, 200],
    'max_depth': [10, 20, None]
}


# Create GridSearchCV for Random Forest.
#
# random_state=2:
#   Ensures reproducible results.
#
# cv=3:
#   Uses 3-fold cross-validation.
#
# scoring='f1_weighted':
#   Optimizes weighted F1-score.
#
# n_jobs=-1:
#   Runs computation using all CPU cores.
rf_search = GridSearchCV(
    RandomForestClassifier(random_state=2),
    rf_grid,
    cv=3,
    scoring='f1_weighted',
    n_jobs=-1
)


# Train Random Forest models with different parameter combinations.
rf_search.fit(X_train, y_train)


# Retrieve the best Random Forest model.
rfc = rf_search.best_estimator_


# Display the best Random Forest parameters.
print("Best RF params:", rf_search.best_params_)



# ============================
# Update Model Dictionary
# ============================

# Replace the original Logistic Regression and Random Forest
# models with their optimized versions.
#
# These tuned models will now be used for final evaluation.
clfs['LR'] = lrc
clfs['RF'] = rfc

In [ ]:
# %% [13] TRAIN ALL MODELS + FULL METRICS
# -----------------------------------------------------------------------------

# Define a function to train a classifier and calculate evaluation metrics.
#
# Inputs:
#   clf      -> Machine learning classifier
#   X_train  -> Training feature data
#   y_train  -> Training labels
#   X_test   -> Testing feature data
#   y_test   -> Testing true labels
#
# Output:
#   Returns Accuracy, Precision, Recall, F1-score, and ROC-AUC
def train_classifier(clf, X_train, y_train, X_test, y_test):

    # Train the model using training data.
    clf.fit(X_train, y_train)


    # Predict class labels for unseen test data.
    y_pred = clf.predict(X_test)


    # Calculate Accuracy:
    # Percentage of total predictions that are correct.
    accuracy = accuracy_score(y_test, y_pred)


    # Calculate Precision:
    # Measures how many predicted positive samples are actually positive.
    #
    # average='weighted':
    # Calculates metrics for each class and averages them
    # based on the number of samples in each class.
    precision = precision_score(
        y_test,
        y_pred,
        average='weighted'
    )


    # Calculate Recall:
    # Measures how many actual positive samples were correctly identified.
    recall = recall_score(
        y_test,
        y_pred,
        average='weighted'
    )


    # Calculate F1-score:
    # Harmonic mean of Precision and Recall.
    # Useful when class distribution is not perfectly balanced.
    f1 = f1_score(
        y_test,
        y_pred,
        average='weighted'
    )


    # Calculate ROC-AUC score.
    #
    # predict_proba() gives probability estimates for each class.
    # [:,1] selects probability of the positive class (target=1).
    #
    # Some algorithms do not support probability prediction,
    # therefore return NaN for those models.
    auc = (
        roc_auc_score(
            y_test,
            clf.predict_proba(X_test)[:, 1]
        )
        if hasattr(clf, "predict_proba")
        else np.nan
    )


    # Return all calculated performance metrics.
    return accuracy, precision, recall, f1, auc



# Create a dictionary to store evaluation results
# for every machine learning algorithm.
results = {
    'Algorithm': [],
    'Accuracy': [],
    'Precision': [],
    'Recall': [],
    'F1': [],
    'ROC-AUC': []
}



# Train and evaluate every classifier stored in 'clfs'.
#
# name -> model name
# clf  -> classifier object
for name, clf in clfs.items():

    try:

        # Train model and calculate all evaluation metrics.
        acc, prec, rec, f1, auc = train_classifier(
            clf,
            X_train,
            y_train,
            X_test,
            y_test
        )


        # Store model name.
        results['Algorithm'].append(name)

        # Store calculated accuracy.
        results['Accuracy'].append(acc)

        # Store precision score.
        results['Precision'].append(prec)

        # Store recall score.
        results['Recall'].append(rec)

        # Store F1-score.
        results['F1'].append(f1)

        # Store ROC-AUC score.
        results['ROC-AUC'].append(auc)


    # If any model fails, display error
    # and continue with remaining models.
    except Exception as e:
        print(f"{name} failed: {e}")



# Convert results dictionary into a Pandas DataFrame.
# This creates a performance comparison table.
performance_df = pd.DataFrame(results)


# Sort models based on F1-score.
# Highest performing model appears at the top.
#
# reset_index(drop=True):
# Removes the old index after sorting.
performance_df = (
    performance_df
    .sort_values('F1', ascending=False)
    .reset_index(drop=True)
)


# Display the final model comparison table.
display(performance_df)


# Save the performance table as a CSV file.
# This can be used later for reports, documentation, or visualization.
performance_df.to_csv(
    'model_performance_table.csv',
    index=False
)

In [ ]:
# %% [14] ENSEMBLE — VOTING + STACKING
# -----------------------------------------------------------------------------
voting = VotingClassifier(estimators=[('LR', lrc), ('RF', rfc)], voting='soft')
voting.fit(X_train, y_train)
y_pred_vote = voting.predict(X_test)
print("Voting  -> Acc:", accuracy_score(y_test, y_pred_vote),
      "F1:", f1_score(y_test, y_pred_vote, average='weighted'))

stack = StackingClassifier(
    estimators=[('LR', lrc), ('RF', rfc)],
    final_estimator=RandomForestClassifier(n_estimators=100, random_state=2)
)
stack.fit(X_train, y_train)
y_pred_stack = stack.predict(X_test)
print("Stacking -> Acc:", accuracy_score(y_test, y_pred_stack),
      "F1:", f1_score(y_test, y_pred_stack, average='weighted'))

In [ ]:
# %% [15] SELECT BEST MODEL AUTOMATICALLY
# -----------------------------------------------------------------------------
candidates = {
    'Voting': (voting, f1_score(y_test, y_pred_vote, average='weighted')),
    'Stacking': (stack, f1_score(y_test, y_pred_stack, average='weighted')),
    performance_df.iloc[0]['Algorithm']: (clfs[performance_df.iloc[0]['Algorithm']],
                                           performance_df.iloc[0]['F1'])
}
best_name = max(candidates, key=lambda k: candidates[k][1])
best_model = candidates[best_name][0]
print(f"\nBest model selected: {best_name} (F1 = {candidates[best_name][1]:.4f})")

y_pred_best = best_model.predict(X_test)



In [ ]:
# %% [16] CONFUSION MATRIX + CLASSIFICATION REPORT
# -----------------------------------------------------------------------------
print(classification_report(y_test, y_pred_best, target_names=['Fake (CG)', 'Real (OR)']))

cm = confusion_matrix(y_test, y_pred_best)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Fake (CG)', 'Real (OR)'], yticklabels=['Fake (CG)', 'Real (OR)'])
plt.xlabel('Predicted'); plt.ylabel('True'); plt.title(f'Confusion Matrix — {best_name}')
plt.savefig('confusion_matrix.png', bbox_inches='tight', dpi=300) # <-- ADD THIS LINE
plt.show()


In [ ]:
# %% [17] ROC-AUC COMPARISON ACROSS MODELS
# -----------------------------------------------------------------------------
plt.figure(figsize=(8, 6))
for name, clf in clfs.items():
    if hasattr(clf, "predict_proba"):
        y_prob = clf.predict_proba(X_test)[:, 1]
        fpr, tpr, _ = roc_curve(y_test, y_prob)
        auc = roc_auc_score(y_test, y_prob)
        plt.plot(fpr, tpr, label=f"{name} (AUC={auc:.3f})")
plt.plot([0, 1], [0, 1], 'k--', label='Random Chance')
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title('ROC-AUC Comparison'); plt.legend();
plt.savefig('roc_auc_curve.png', bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
# %% [18] METRIC COMPARISON BAR CHARTS
# -----------------------------------------------------------------------------
performance_df.set_index('Algorithm')[['Accuracy', 'Precision', 'Recall', 'F1']].plot(
    kind='bar', figsize=(10, 5), title="Model Comparison Across Metrics")
plt.ylabel("Score"); plt.xticks(rotation=0);
plt.savefig('metrics_bar_chart.png', bbox_inches='tight', dpi=300) # <-- ADD THIS LINE
plt.show()


In [ ]:

# %% [19] FAST EXPLAINABILITY — LOGISTIC REGRESSION COEFFICIENTS
# -----------------------------------------------------------------------------
feature_names = np.array(list(tfidf.get_feature_names_out()) + meta_cols)
coefs = lrc.coef_[0]

top_fake_idx = np.argsort(coefs)[:15]
top_real_idx = np.argsort(coefs)[-15:]

plt.figure(figsize=(10, 6))
plt.barh(feature_names[top_fake_idx], coefs[top_fake_idx], color='red', label='Fake (CG) signal')
plt.barh(feature_names[top_real_idx], coefs[top_real_idx], color='green', label='Real (OR) signal')
plt.xlabel('Coefficient Weight'); plt.title('Top Predictors: Fake vs Real Reviews')
plt.legend(); plt.tight_layout()
plt.savefig('logistic_regression_coefficients.png', bbox_inches='tight', dpi=300)
plt.show()
files.download('logistic_regression_coefficients.png')

In [ ]:
# %% [20] SHAP EXPLAINABILITY (Fix for Additivity and Waterfall Errors)
# -----------------------------------------------------------------------------
import shap

# 1. Take a small sample to keep the calculation fast
sample_size = min(50, X_test.shape[0])
X_test_sample = X_test[:sample_size].toarray()

# 2. Initialize the Explainer
explainer = shap.TreeExplainer(rfc)

# 3. CRITICAL FIX: Add check_additivity=False and approximate=True
print("Calculating SHAP values...")
shap_values = explainer.shap_values(
    X_test_sample,
    approximate=True,
    check_additivity=False
)

# 4. Handle SHAP formatting across different library versions
if isinstance(shap_values, list):
    shap_vals_for_plot = shap_values[1]        # It's a list: take the Real (OR) class
elif len(shap_values.shape) == 3:
    shap_vals_for_plot = shap_values[:, :, 1]  # It's a 3D matrix: slice the Real (OR) class
else:
    shap_vals_for_plot = shap_values           # Fallback

# 5. Summary Plot
plt.figure(figsize=(10, 6))
shap.summary_plot(
    shap_vals_for_plot,
    X_test_sample,
    feature_names=feature_names,
    show=False # <-- Change to False so we can save it first
)
plt.savefig('shap_summary_plot.png', bbox_inches='tight', dpi=300) # <-- ADD THIS LINE
# files.download('shap_summary_plot.png')
plt.show()

# 6. Waterfall Plot (for the first sample)
print("Generating Waterfall plot...")
base_value = explainer.expected_value[1] if isinstance(explainer.expected_value, (list, np.ndarray)) else explainer.expected_value

# Generate the plot but hold it in memory (show=False)
shap.plots._waterfall.waterfall_legacy(
    base_value,
    shap_vals_for_plot[0],
    feature_names=feature_names,
    show=False # <-- Added this!
)

# Save, show, and download
plt.savefig('shap_waterfall_plot.png', bbox_inches='tight', dpi=300)
plt.show()

# files.download('shap_waterfall_plot.png')

In [ ]:
# %% [20.1] LIME EXPLAINABILITY (SHAP vs LIME Comparison)
# -----------------------------------------------------------------------------
!pip install -q lime
import lime
import lime.lime_tabular

print("Initializing LIME Explainer...")
# LIME requires dense arrays, so we take a small sample of the training data
X_train_dense = X_train[:100].toarray()

lime_explainer = lime.lime_tabular.LimeTabularExplainer(
    training_data=X_train_dense,
    feature_names=feature_names,
    class_names=['Fake (CG)', 'Real (OR)'],
    mode='classification'
)

# We explain the same test sample we used for the SHAP waterfall plot
sample_idx = 0
X_test_dense_sample = X_test_sample[sample_idx]

print("Generating LIME Explanation...")
# LIME needs a function that takes data and returns probabilities
def predict_fn(x):
    return best_model.predict_proba(x)

lime_exp = lime_explainer.explain_instance(
    data_row=X_test_dense_sample,
    predict_fn=predict_fn,
    num_features=10
)

# Plot, Save, and Download
lime_fig = lime_exp.as_pyplot_figure()
plt.title("LIME Explanation - Top Features")
plt.savefig('lime_explanation.png', bbox_inches='tight', dpi=300)
plt.show()

from google.colab import files
# files.download('lime_explanation.png')

In [ ]:
# %% [21] SAVE BEST MODEL + PREPROCESSING OBJECTS
# -----------------------------------------------------------------------------
import pickle
from google.colab import files

# 1. Save the files to Colab's temporary cloud storage
pickle.dump(best_model, open('best_model.pkl', 'wb'))
pickle.dump(tfidf, open('vectorizer.pkl', 'wb'))
pickle.dump(scaler, open('scaler.pkl', 'wb'))
pickle.dump(le_category, open('category_encoder.pkl', 'wb'))
print(f"Saved best model: {best_name}")

# 2. Automatically download them to your physical computer
# files.download('best_model.pkl')
# files.download('vectorizer.pkl')
# files.download('scaler.pkl')
# files.download('category_encoder.pkl')

In [ ]:
# %% [21.1] RESEARCH ANALYSIS: TF-IDF ONLY vs FEATURE FUSION
# -----------------------------------------------------------------------------
print("Comparing Text-Only (TF-IDF) vs Feature Fusion (Text + Metadata)...")

# 1. Isolate only the text features (first 5000 columns)
num_text_features = X_text.shape[1]
X_train_text = X_train[:, :num_text_features]
X_test_text = X_test[:, :num_text_features]

# 2. Train a baseline Logistic Regression on TEXT ONLY
lr_text_only = LogisticRegression(solver='liblinear', max_iter=1000)
lr_text_only.fit(X_train_text, y_train)
y_pred_text = lr_text_only.predict(X_test_text)
f1_text = f1_score(y_test, y_pred_text, average='weighted')

# 3. Get the F1 score of our fused Logistic Regression
f1_fused = performance_df[performance_df['Algorithm'] == 'LR']['F1'].values[0]

# 4. Compare and Plot
ablation_df = pd.DataFrame({
    'Method': ['Baseline (TF-IDF Text Only)', 'Proposed (Feature Fusion)'],
    'F1-Score': [f1_text, f1_fused]
})
display(ablation_df)

plt.figure(figsize=(7, 4))
sns.barplot(x='Method', y='F1-Score', data=ablation_df, palette='viridis')
plt.ylim(min(f1_text, f1_fused) - 0.02, max(f1_text, f1_fused) + 0.02)
plt.title("Impact of Feature Fusion on Model Performance")
plt.ylabel("F1-Score")
plt.savefig('ablation_study_graph.png', bbox_inches='tight', dpi=300) # <-- ADD THIS LINE
# files.download('ablation_study_graph.png')
plt.show()

In [ ]:
# %% [21.2] RESEARCH ANALYSIS: CATEGORY-WISE PERFORMANCE
# -----------------------------------------------------------------------------
cat_results = []

# Predict on the entire dataset to evaluate categories
df['predicted_target'] = best_model.predict(X)

for cat in df['category'].unique():
    subset = df[df['category'] == cat]
    if len(subset) > 50:  # Only evaluate categories with meaningful sample sizes
        acc = accuracy_score(subset['target'], subset['predicted_target'])
        cat_results.append({'Category': cat, 'Accuracy': acc, 'Total Reviews': len(subset)})

cat_df = pd.DataFrame(cat_results).sort_values(by='Accuracy', ascending=False)
display(cat_df.head(10))

plt.figure(figsize=(10, 5))
sns.barplot(x='Accuracy', y='Category', data=cat_df.head(10), palette='mako')
plt.title("Top 10 Categories by Fake Review Detection Accuracy")
plt.xlabel("Accuracy")
plt.xlim(0.7, 1.0)

# Save and show the image
plt.savefig('category_performance.png', bbox_inches='tight', dpi=300)
plt.show()

# Download the image and the table
from google.colab import files
# files.download('category_performance.png')

cat_df.to_csv('category_performance.csv', index=False)
# files.download('category_performance.csv')

In [ ]:
# %% [21.3] RESEARCH ANALYSIS: ERROR ANALYSIS (Misclassifications)
# -----------------------------------------------------------------------------
from google.colab import files
import numpy as np

# Create a boolean mask of where the model was wrong
error_mask = (df['target'] != df['predicted_target']).values

# Filter for reviews where the model's prediction was wrong
mistakes = df[error_mask].copy()

# Map numeric targets back to readable labels for the report
label_map = {1: 'Real (OR)', 0: 'Fake (CG)'}
mistakes['Actual'] = mistakes['target'].map(label_map)
mistakes['Predicted'] = mistakes['predicted_target'].map(label_map)

# FIX: Use the boolean mask to slice 'X' instead of mistakes.index
mistakes['Confidence'] = np.max(best_model.predict_proba(X[error_mask]), axis=1)

print(f"Total misclassifications in dataset: {len(mistakes)} out of {len(df)}")
print("--- Sample Misclassified Reviews ---")

# Display top 5 mistakes with highest confidence (where the model was very wrong)
top_mistakes = mistakes.sort_values(by='Confidence', ascending=False).head(5)

for idx, row in top_mistakes.iterrows():
    print(f"\nActual: {row['Actual']} | Predicted: {row['Predicted']} (Confidence: {row['Confidence']:.2f})")
    print(f"Review: {row['text'][:250]}...")

# --- FIXED SAVE & DOWNLOAD SECTION ---

# 1. Save and download the error analysis data
mistakes.to_csv('misclassified_reviews.csv', index=False)
# files.download('misclassified_reviews.csv')

# 2. Save and download the overall model performance (from Cell 13)
# performance_df.to_csv('model_performance_metrics.csv', index=False)
# files.download('model_performance_metrics.csv')

In [ ]:
# %% [21.4] RESEARCH ANALYSIS: ROBUSTNESS TESTING (Adversarial Typos)
# -----------------------------------------------------------------------------
import random

def add_typo(text):
    """Simulates user typos by randomly duplicating or swapping a character."""
    if len(text) < 5: return text
    words = text.split()
    if not words: return text

    # Pick a random word to alter
    idx = random.randint(0, len(words)-1)
    word = list(words[idx])

    if len(word) > 3:
        # 50% chance to duplicate a letter, 50% to swap adjacent letters
        if random.random() > 0.5:
            char_idx = random.randint(1, len(word)-2)
            word.insert(char_idx, word[char_idx])
        else:
            char_idx = random.randint(0, len(word)-2)
            word[char_idx], word[char_idx+1] = word[char_idx+1], word[char_idx]

    words[idx] = "".join(word)
    return " ".join(words)

print("Running Robustness Test on 500 random samples...")
# Take a sample of 500 reviews
robust_df = df.sample(500, random_state=42).copy()

# 1. Original Predictions (Clean Text)
robust_X_t = tfidf.transform(robust_df['transformed_text'])
robust_X_m = scaler.transform(robust_df[meta_cols])
robust_X = hstack([robust_X_t, robust_X_m]).tocsr()
original_preds = best_model.predict(robust_X)
original_acc = accuracy_score(robust_df['target'], original_preds)

# 2. Apply Typos (Noisy Text)
robust_df['noisy_text'] = robust_df['text'].apply(add_typo)
robust_df['noisy_transformed'] = robust_df['noisy_text'].apply(transform_text)

noisy_X_t = tfidf.transform(robust_df['noisy_transformed'])
noisy_X = hstack([noisy_X_t, robust_X_m]).tocsr()
noisy_preds = best_model.predict(noisy_X)
noisy_acc = accuracy_score(robust_df['target'], noisy_preds)

print(f"\n✅ Original Accuracy (Clean Text): {original_acc * 100:.2f}%")
print(f"⚠️ Robustness Accuracy (With Typos): {noisy_acc * 100:.2f}%")
print(f"📉 Performance Drop: {(original_acc - noisy_acc) * 100:.2f}%")

# Show an example of the modification
sample_idx = robust_df.index[0]
print("\n--- Example Modification ---")
print("Original:", robust_df.loc[sample_idx, 'text'])
print("Modified:", robust_df.loc[sample_idx, 'noisy_text'])

In [ ]:
# %% [22] LIVE DEMO WITH GRADIO (Explanation Dashboard & Trust Score)
# -----------------------------------------------------------------------------
import gradio as gr

# Create a dictionary of word weights from Logistic Regression for fast highlighting
word_coefs = dict(zip(tfidf.get_feature_names_out(), lrc.coef_[0][:len(tfidf.get_feature_names_out())]))

def predict_review_dashboard(review_text, rating, category):
    # 1. Process Text
    transformed = transform_text(review_text)
    X_t = tfidf.transform([transformed])

    # 2. Process Metadata
    try:
        cat_encoded = le_category.transform([category])[0]
    except ValueError:
        cat_encoded = 0

    char_count = len(review_text)
    word_count = len(review_text.split())
    sent_count = len(nltk.sent_tokenize(review_text)) if review_text.strip() else 0
    cap_ratio = sum(1 for c in review_text if c.isupper()) / (len(review_text) + 1)
    excl_count = review_text.count('!')

    X_m = scaler.transform([[float(rating), cat_encoded, char_count, word_count,
                              sent_count, cap_ratio, excl_count]])

    # 3. Predict using the best ensemble model
    X_input = hstack([X_t, X_m]).tocsr()
    prob = best_model.predict_proba(X_input)[0]

    # 4. Trust Score (Idea #4)
    trust_score = round(float(prob[1]) * 100, 1)

    if trust_score >= 70:
        verdict = f"✅ Highly Trusted ({trust_score}/100)"
    elif trust_score >= 40:
        verdict = f"⚠️ Suspicious ({trust_score}/100)"
    else:
        verdict = f"🚨 Likely Fake/Bot ({trust_score}/100)"

    probabilities = {"Fake (Computer Generated)": float(prob[0]), "Real (Original)": float(prob[1])}

    # 5. Explanation Dashboard: Highlighting Words & Metadata (Idea #3)
    highlighted_html = "<div style='line-height: 1.6; font-size: 16px; border: 1px solid #e0e0e0; padding: 10px; border-radius: 5px;'>"
    words = review_text.split()

    for original_word in words:
        clean_word = transform_text(original_word).strip()
        weight = word_coefs.get(clean_word, 0)

        # Color code based on word impact
        if weight > 0.5:
            color = "#d4edda" # Green (Real signal)
            tooltip = f"Real Signal (+{weight:.2f})"
        elif weight < -0.5:
            color = "#f8d7da" # Red (Fake signal)
            tooltip = f"Fake Signal ({weight:.2f})"
        else:
            color = "transparent"
            tooltip = "Neutral"

        if color != "transparent":
            highlighted_html += f"<span style='background-color: {color}; padding: 2px 4px; border-radius: 3px; cursor: help;' title='{tooltip}'>{original_word}</span> "
        else:
            highlighted_html += f"{original_word} "

    highlighted_html += "</div>"

    # Metadata explanation breakdown
    meta_html = "<ul style='font-size: 14px; margin-top: 10px;'>"
    if excl_count > 3: meta_html += "<li>🔴 <b>High exclamation count:</b> Fake reviews often overuse punctuation.</li>"
    if cap_ratio > 0.1: meta_html += "<li>🔴 <b>High ALL CAPS ratio:</b> Over-emotional text is a bot indicator.</li>"
    if rating == 1.0 or rating == 5.0: meta_html += "<li>🟡 <b>Extreme rating:</b> Bots rarely leave nuanced 2, 3, or 4-star reviews.</li>"
    if len(words) < 5: meta_html += "<li>🟡 <b>Suspiciously short:</b> Lack of descriptive detail.</li>"
    if meta_html == "<ul style='font-size: 14px; margin-top: 10px;'>": meta_html = "<ul><li>🟢 Metadata (length, caps, punctuation) looks natural.</li>"
    meta_html += "</ul>"

    final_explanation = f"<h4>Highlighted Text Analysis:</h4>{highlighted_html}<h4>Metadata Analysis:</h4>{meta_html}"

    return verdict, probabilities, final_explanation

demo = gr.Interface(
    fn=predict_review_dashboard,
    inputs=[
        gr.Textbox(lines=4, placeholder="Paste a review here to test it...", label="Review Text"),
        gr.Slider(1, 5, value=5, step=1, label="Star Rating"),
        gr.Dropdown(choices=list(le_category.classes_), label="Product Category", value=le_category.classes_[0])
    ],
    outputs=[
        gr.Textbox(label="Review Trust Score"),
        gr.Label(num_top_classes=2, label="Raw Probabilities"),
        gr.HTML(label="Explanation Dashboard (Red = Fake, Green = Real)")
    ],
    title="TrustScore: Explainable Fake Review Detection",
    description="Generates a 0-100 Trust Score based on text patterns and metadata. Hover over colored words to see their exact impact on the AI's decision."
)
demo.launch(share=True)

In [ ]:
!pwd
!ls -la